In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
import json
import os

In [13]:
df = pd.read_csv('C:/Users/user/JobGenie/Student/Dataset/student_salary_dataset.csv')
print(f"✅ Loaded dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Columns: {df.columns.tolist()}")

✅ Loaded dataset: 5000 rows × 14 columns
   Columns: ['college_tier', 'cgpa', 'internships', 'github_projects', 'backlogs', 'hackathons', 'certifications', 'skills', 'advanced_skills', 'intermediate_skills', 'basic_skills', 'aptitude_score', 'job_role', 'salary_lpa']


In [14]:
df.drop(columns=['skills'], inplace=True)
print("\n✅ STEP 1: Dropped raw 'skills' column (info preserved in skill count columns)")


✅ STEP 1: Dropped raw 'skills' column (info preserved in skill count columns)


In [15]:
tier_mapping = {'Tier 1': 3, 'Tier 2': 2, 'Tier 3': 1}
df['college_tier_encoded'] = df['college_tier'].map(tier_mapping)
print("\n✅ STEP 2: Encoded college_tier (Tier 1=3, Tier 2=2, Tier 3=1)")
print(f"   Mapping: {tier_mapping}")


✅ STEP 2: Encoded college_tier (Tier 1=3, Tier 2=2, Tier 3=1)
   Mapping: {'Tier 1': 3, 'Tier 2': 2, 'Tier 3': 1}


In [16]:
le = LabelEncoder()
df['job_role_encoded'] = le.fit_transform(df['job_role'])
role_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("\n✅ STEP 3: Encoded job_role using LabelEncoder")
print(f"   Mapping: {role_mapping}")


✅ STEP 3: Encoded job_role using LabelEncoder
   Mapping: {'Backend Developer': 0, 'Data Analyst': 1, 'Data Scientist': 2, 'DevOps Engineer': 3, 'Junior Developer': 4, 'ML Engineer': 5, 'Software Engineer': 6, 'Web Developer': 7}


In [17]:
df['aptitude_score'] = df['aptitude_score'].round(1)
print("\n✅ STEP 4: Rounded aptitude_score to 1 decimal place")


✅ STEP 4: Rounded aptitude_score to 1 decimal place


In [18]:
original_skew = df['salary_lpa'].skew()
df['salary_log'] = np.log1p(df['salary_lpa'])  # log1p = log(1+x), safe for 0
log_skew = df['salary_log'].skew()
print(f"\n✅ STEP 5: Salary skewness check")
print(f"   Original salary skewness : {original_skew:.4f}")
print(f"   Log-transformed skewness : {log_skew:.4f}")
print(f"   → We will train on original salary_lpa (Random Forest handles skew well)")
print(f"   → salary_log column kept as reference")


✅ STEP 5: Salary skewness check
   Original salary skewness : 0.9436
   Log-transformed skewness : 0.0017
   → We will train on original salary_lpa (Random Forest handles skew well)
   → salary_log column kept as reference


In [19]:
role_counts = df['job_role'].value_counts()
print(f"\n✅ STEP 6: Class imbalance check for job_role")
print(f"   Most common  : {role_counts.idxmax()} ({role_counts.max()} rows)")
print(f"   Least common : {role_counts.idxmin()} ({role_counts.min()} rows)")
print(f"   Imbalance ratio: {role_counts.max()/role_counts.min():.1f}x")
print(f"   → Will use class_weight='balanced' during model training")


✅ STEP 6: Class imbalance check for job_role
   Most common  : Software Engineer (1138 rows)
   Least common : Backend Developer (210 rows)
   Imbalance ratio: 5.4x
   → Will use class_weight='balanced' during model training


In [20]:
print("\n✅ STEP 7: Final data types check")
print(df.dtypes)


✅ STEP 7: Final data types check
college_tier             object
cgpa                    float64
internships               int64
github_projects           int64
backlogs                  int64
hackathons                int64
certifications            int64
advanced_skills           int64
intermediate_skills       int64
basic_skills              int64
aptitude_score          float64
job_role                 object
salary_lpa              float64
college_tier_encoded      int64
job_role_encoded          int32
salary_log              float64
dtype: object


In [21]:
FEATURE_COLUMNS = [
    'college_tier_encoded',
    'cgpa',
    'internships',
    'github_projects',
    'backlogs',
    'hackathons',
    'certifications',
    'advanced_skills',
    'intermediate_skills',
    'basic_skills',
    'aptitude_score'
]
 
TARGET_SALARY   = 'salary_lpa'
TARGET_ROLE     = 'job_role_encoded'
 
print(f"\n✅ STEP 8: Feature columns defined")
print(f"   Features ({len(FEATURE_COLUMNS)}): {FEATURE_COLUMNS}")
print(f"   Target 1 (regression)    : {TARGET_SALARY}")
print(f"   Target 2 (classification): {TARGET_ROLE}")


✅ STEP 8: Feature columns defined
   Features (11): ['college_tier_encoded', 'cgpa', 'internships', 'github_projects', 'backlogs', 'hackathons', 'certifications', 'advanced_skills', 'intermediate_skills', 'basic_skills', 'aptitude_score']
   Target 1 (regression)    : salary_lpa
   Target 2 (classification): job_role_encoded


In [22]:
print(f"\n✅ STEP 9: Final cleaned dataset shape: {df.shape}")
print(f"\n   Sample (first 3 rows):")
print(df[FEATURE_COLUMNS + [TARGET_SALARY, TARGET_ROLE, 'job_role', 'college_tier']].head(3).to_string())


✅ STEP 9: Final cleaned dataset shape: (5000, 16)

   Sample (first 3 rows):
   college_tier_encoded  cgpa  internships  github_projects  backlogs  hackathons  certifications  advanced_skills  intermediate_skills  basic_skills  aptitude_score  salary_lpa  job_role_encoded           job_role college_tier
0                     2  7.60            2                1         1           1               3                0                    1             0            52.5       10.23                 6  Software Engineer       Tier 2
1                     2  5.89            1                2         0           1               2                2                    0             0            46.5        6.03                 7      Web Developer       Tier 2
2                     3  5.62            0                4         1           0               1                1                    2             1            45.2        5.93                 4   Junior Developer       Tier 1


In [25]:
# 1. Cleaned CSV
df.to_csv('C:/Users/user/JobGenie/Student/Notebook/student_cleaned.csv', index=False)
print("\n✅ Saved: student_cleaned.csv")
 
# 2. Feature columns list (for Flask API reference)
meta = {
    'feature_columns': FEATURE_COLUMNS,
    'target_salary': TARGET_SALARY,
    'target_role': TARGET_ROLE,
    'tier_mapping': tier_mapping,
    'role_mapping': {k: int(v) for k, v in role_mapping.items()},
    'role_classes': list(le.classes_)
}
with open('C:/Users/user/JobGenie/Student/Notebook/cleaning_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print("✅ Saved: cleaning_meta.json (encoders + feature list)")
 
print("\n" + "="*55)
print("  DATA CLEANING COMPLETE")
print("="*55)
print(f"  Input  : 5000 rows × 14 columns")
print(f"  Output : 5000 rows × {df.shape[1]} columns")
print(f"  Changes made:")
print(f"    ✓ Dropped raw 'skills' string column")
print(f"    ✓ Encoded college_tier (ordinal: 1/2/3)")
print(f"    ✓ Encoded job_role (LabelEncoder)")
print(f"    ✓ Rounded aptitude_score to 1 decimal")
print(f"    ✓ Added salary_log for reference")
print(f"    ✓ Saved feature column list for API use")
print(f"    ✓ No rows dropped — all 5000 kept")
print("="*55)


✅ Saved: student_cleaned.csv
✅ Saved: cleaning_meta.json (encoders + feature list)

  DATA CLEANING COMPLETE
  Input  : 5000 rows × 14 columns
  Output : 5000 rows × 16 columns
  Changes made:
    ✓ Dropped raw 'skills' string column
    ✓ Encoded college_tier (ordinal: 1/2/3)
    ✓ Encoded job_role (LabelEncoder)
    ✓ Rounded aptitude_score to 1 decimal
    ✓ Added salary_log for reference
    ✓ Saved feature column list for API use
    ✓ No rows dropped — all 5000 kept
